In [1]:
# CEFR Level Classification Model - Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*60)
print("CEFR TEXT CLASSIFICATION MODEL - CUSTOM ARCHITECTURE")
print("="*60)

CEFR TEXT CLASSIFICATION MODEL - CUSTOM ARCHITECTURE


In [2]:
# Load Dataset
print("="*60)
print("LOADING DATASET")
print("="*60)

df = pd.read_csv('/kaggle/input/datasets/ahmedsameh72/cefr-gradproj/cefr_large_dataset-2.csv')

print(f"\nDataset loaded!")
print(f"Total samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()

LOADING DATASET

Dataset loaded!
Total samples: 151816
Columns: ['text', 'label', 'source']

First few rows:


,text,label,source
0,I just had my third interview with a company. ...,B2,reddit
1,Just wondering what made you decide you needed...,B1,reddit
2,Some background: I'm from California and my gi...,B2,reddit
3,Hey guys:\n\nHere's a rough draft I'm submitti...,C2,reddit
4,"**Section A:** *It's BACON!*\n\nAll ministers,...",B2,reddit


In [3]:
# Data Preprocessing - Step 1: Basic Cleaning
print("="*60)
print("STEP 1: BASIC DATA CLEANING")
print("="*60)

print(f"\nOriginal dataset size: {len(df)}")

df_clean = df.copy()

print(f"Null values per column:\n{df_clean.isnull().sum()}")
df_clean = df_clean.dropna()
print(f"After removing nulls: {len(df_clean)}")

duplicates = df_clean['text'].duplicated().sum()
df_clean = df_clean.drop_duplicates(subset=['text'], keep='first')
print(f"Duplicates removed: {duplicates}")
print(f"After removing duplicates: {len(df_clean)}")

df_clean['text'] = df_clean['text'].astype(str).str.strip()

df_clean['text_length'] = df_clean['text'].apply(len)
df_clean['word_count'] = df_clean['text'].apply(lambda x: len(x.split()))

print(f"\nFinal cleaned dataset: {len(df_clean)} samples")
print("="*60)

STEP 1: BASIC DATA CLEANING

Original dataset size: 151816
Null values per column:
text      0
label     0
source    0
dtype: int64
After removing nulls: 151816
Duplicates removed: 0
After removing duplicates: 151816

Final cleaned dataset: 151816 samples


In [4]:
# Data Preprocessing - Step 2: Analyze Class Distribution
print("="*60)
print("STEP 2: CLASS DISTRIBUTION ANALYSIS")
print("="*60)

print("\nCEFR Level Distribution:")
print("-"*60)
label_counts = df_clean['label'].value_counts().sort_index()
label_percentages = (df_clean['label'].value_counts(normalize=True).sort_index() * 100)

distribution_df = pd.DataFrame({
    'Count': label_counts,
    'Percentage': label_percentages.round(2)
})
print(distribution_df)

print("\n" + "="*60)
print("CLASS IMBALANCE DETECTED!")
print("="*60)
print("This will be handled using class weights during training.")
print("="*60)

STEP 2: CLASS DISTRIBUTION ANALYSIS

CEFR Level Distribution:
------------------------------------------------------------
       Count  Percentage
label                   
A1       167        0.11
A2     40000       26.35
B1     40000       26.35
B2     40000       26.35
C1     21286       14.02
C2     10363        6.83

CLASS IMBALANCE DETECTED!
This will be handled using class weights during training.


In [5]:
# Data Preprocessing - Step 3: Filter Outliers
print("="*60)
print("STEP 3: OUTLIER ANALYSIS")
print("="*60)

print(f"\nCurrent dataset size: {len(df_clean)}")
print(f"\nWord count statistics:")
print(df_clean['word_count'].describe())

print("\n" + "-"*60)
print("SHORT TEXTS:")
for threshold in [5, 10, 20]:
    short_count = len(df_clean[df_clean['word_count'] < threshold])
    print(f"  < {threshold} words: {short_count} ({short_count/len(df_clean)*100:.2f}%)")

print("\nLONG TEXTS:")
for threshold in [500, 750, 1000]:
    long_count = len(df_clean[df_clean['word_count'] > threshold])
    print(f"  > {threshold} words: {long_count} ({long_count/len(df_clean)*100:.2f}%)")

print("\n" + "-"*60)
print("Applying default filter: removing texts < 10 words")
min_words = 10
df_processed = df_clean[df_clean['word_count'] >= min_words].copy()
print(f"Samples removed: {len(df_clean) - len(df_processed)}")
print(f"Final dataset size: {len(df_processed)}")
print("="*60)

STEP 3: OUTLIER ANALYSIS

Current dataset size: 151816

Word count statistics:
count    151816.000000
mean        127.495639
std         127.092042
min           1.000000
25%          42.000000
50%          84.000000
75%         165.000000
max        1322.000000
Name: word_count, dtype: float64

------------------------------------------------------------
SHORT TEXTS:
  < 5 words: 25 (0.02%)
  < 10 words: 1686 (1.11%)
  < 20 words: 10617 (6.99%)

LONG TEXTS:
  > 500 words: 3842 (2.53%)
  > 750 words: 235 (0.15%)
  > 1000 words: 19 (0.01%)

------------------------------------------------------------
Applying default filter: removing texts < 10 words
Samples removed: 1686
Final dataset size: 150130


In [6]:
# Data Preprocessing - Step 4: Prepare Features and Labels
print("="*60)
print("STEP 4: PREPARE FEATURES AND LABELS")
print("="*60)

X = df_processed['text'].values
y = df_processed['label'].values

print(f"\nFeatures (X): {X.shape}")
print(f"Labels (y): {y.shape}")
print(f"\nUnique labels: {np.unique(y)}")
print(f"Label distribution:\n{pd.Series(y).value_counts().sort_index()}")

from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"\nLabel encoding mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"  {label} -> {i}")

print(f"\nEncoded labels shape: {y_encoded.shape}")
print("="*60)

STEP 4: PREPARE FEATURES AND LABELS

Features (X): (150130,)
Labels (y): (150130,)

Unique labels: ['A1' 'A2' 'B1' 'B2' 'C1' 'C2']
Label distribution:
A1       79
A2    39379
B1    39430
B2    39738
C1    21153
C2    10351
Name: count, dtype: int64

Label encoding mapping:
  A1 -> 0
  A2 -> 1
  B1 -> 2
  B2 -> 3
  C1 -> 4
  C2 -> 5

Encoded labels shape: (150130,)


In [7]:
# Data Preprocessing - Step 5: Train-Test-Validation Split
print("="*60)
print("STEP 5: TRAIN-TEST-VALIDATION SPLIT")
print("="*60)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y_encoded,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print(f"\nDataset splits:")
print(f"  Training set:   {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation set: {len(X_val)} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set:       {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nTraining set label distribution:")
train_dist = pd.Series(y_train).value_counts().sort_index()
for idx, count in train_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_train)*100:.1f}%)")

print(f"\nValidation set label distribution:")
val_dist = pd.Series(y_val).value_counts().sort_index()
for idx, count in val_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_val)*100:.1f}%)")

print(f"\nTest set label distribution:")
test_dist = pd.Series(y_test).value_counts().sort_index()
for idx, count in test_dist.items():
    label_name = label_encoder.classes_[idx]
    print(f"  {label_name}: {count} ({count/len(y_test)*100:.1f}%)")

print("="*60)

STEP 5: TRAIN-TEST-VALIDATION SPLIT

Dataset splits:
  Training set:   108468 samples (72.2%)
  Validation set: 19142 samples (12.8%)
  Test set:       22520 samples (15.0%)

Training set label distribution:
  A1: 57 (0.1%)
  A2: 28451 (26.2%)
  B1: 28489 (26.3%)
  B2: 28710 (26.5%)
  C1: 15283 (14.1%)
  C2: 7478 (6.9%)

Validation set label distribution:
  A1: 10 (0.1%)
  A2: 5021 (26.2%)
  B1: 5027 (26.3%)
  B2: 5067 (26.5%)
  C1: 2697 (14.1%)
  C2: 1320 (6.9%)

Test set label distribution:
  A1: 12 (0.1%)
  A2: 5907 (26.2%)
  B1: 5914 (26.3%)
  B2: 5961 (26.5%)
  C1: 3173 (14.1%)
  C2: 1553 (6.9%)


In [8]:
# Data Preprocessing - Step 6: Compute Class Weights
print("="*60)
print("STEP 6: COMPUTE CLASS WEIGHTS FOR IMBALANCE")
print("="*60)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(enumerate(class_weights))

print("\nClass Weights (higher weight = rarer class):")
print("-"*60)
for class_idx, weight in class_weight_dict.items():
    label_name = label_encoder.classes_[class_idx]
    count = (y_train == class_idx).sum()
    percentage = (count / len(y_train)) * 100
    print(f"  {label_name} (Class {class_idx}): Weight = {weight:.2f} | Count = {count} ({percentage:.2f}%)")

print("\n" + "="*60)
print("Class weights will give higher importance to minority classes")
print("="*60)

STEP 6: COMPUTE CLASS WEIGHTS FOR IMBALANCE

Class Weights (higher weight = rarer class):
------------------------------------------------------------
  A1 (Class 0): Weight = 317.16 | Count = 57 (0.05%)
  A2 (Class 1): Weight = 0.64 | Count = 28451 (26.23%)
  B1 (Class 2): Weight = 0.63 | Count = 28489 (26.26%)
  B2 (Class 3): Weight = 0.63 | Count = 28710 (26.47%)
  C1 (Class 4): Weight = 1.18 | Count = 15283 (14.09%)
  C2 (Class 5): Weight = 2.42 | Count = 7478 (6.89%)

Class weights will give higher importance to minority classes


In [9]:
# Save Preprocessed Data
print("="*60)
print("SAVING PREPROCESSED DATA")
print("="*60)

df_final = df_processed[['text', 'label', 'source']].copy()

output_file = 'cefr_dataset_processed.csv'
df_final.to_csv(output_file, index=False)
print(f"\nPreprocessed data saved to: {output_file}")
print(f"   Total samples: {len(df_final)}")

import pickle
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder saved to: label_encoder.pkl")

np.save('X_train.npy', X_train)
np.save('X_val.npy', X_val)
np.save('X_test.npy', X_test)
np.save('y_train.npy', y_train)
np.save('y_val.npy', y_val)
np.save('y_test.npy', y_test)
print(f"Train/Val/Test splits saved")

# Save class weights
with open('class_weights.pkl', 'wb') as f:
    pickle.dump(class_weight_dict, f)
print(f"Class weights saved to: class_weights.pkl")

print("\n" + "="*60)
print("PREPROCESSING COMPLETE!")
print("="*60)

SAVING PREPROCESSED DATA

Preprocessed data saved to: cefr_dataset_processed.csv
   Total samples: 150130
Label encoder saved to: label_encoder.pkl
Train/Val/Test splits saved
Class weights saved to: class_weights.pkl

PREPROCESSING COMPLETE!


In [10]:
print("="*60)
print("IMPORTING LIBRARIES FOR CUSTOM MODEL")
print("="*60)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import time

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("   No GPU detected. Training will be slower on CPU.")

print("="*60)

IMPORTING LIBRARIES FOR CUSTOM MODEL

🖥️  Device: cuda
   GPU: Tesla T4
   Memory: 14.56 GB


In [11]:
# Simple Tokenizer Class
class SimpleTokenizer:
    def __init__(self, vocab_size=10000, max_length=512):
        self.vocab_size = vocab_size
        self.max_length = max_length
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.current_idx = 2

    def build_vocab(self, texts):
        word_freq = {}
        for text in texts:
            words = text.lower().split()
            for word in words:
                word_freq[word] = word_freq.get(word, 0) + 1

        sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
        for word, freq in sorted_words:
            if len(self.word2idx) < self.vocab_size:
                self.word2idx[word] = self.current_idx
                self.idx2word[self.current_idx] = word
                self.current_idx += 1

        print(f"Vocabulary built with {len(self.word2idx)} words")

    def encode(self, text):
        words = text.lower().split()
        indices = []
        for word in words:
            if word in self.word2idx:
                indices.append(self.word2idx[word])
            else:
                indices.append(self.word2idx['<UNK>'])

        if len(indices) < self.max_length:
            indices = indices + [0] * (self.max_length - len(indices))
        else:
            indices = indices[:self.max_length]

        return indices

    def decode(self, indices):
        words = [self.idx2word.get(idx, '<UNK>') for idx in indices if idx != 0]
        return ' '.join(words)

print("Building tokenizer vocabulary...")
tokenizer = SimpleTokenizer(vocab_size=10000, max_length=512)
tokenizer.build_vocab(X_train)
print("Tokenizer ready!")
print(f"   Vocabulary size: {len(tokenizer.word2idx)}")
print(f"   Max sequence length: {tokenizer.max_length}")

Building tokenizer vocabulary...
Vocabulary built with 10000 words
Tokenizer ready!
   Vocabulary size: 10000
   Max sequence length: 512


In [12]:
# Custom Dataset Class for CEFR
class CEFRDatasetCustom(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        input_ids = self.tokenizer.encode(text)

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("="*60)
print("CREATING DATASETS")
print("="*60)

train_dataset = CEFRDatasetCustom(X_train, y_train, tokenizer)
val_dataset = CEFRDatasetCustom(X_val, y_val, tokenizer)
test_dataset = CEFRDatasetCustom(X_test, y_test, tokenizer)

print(f"\nDatasets created:")
print(f"   Training:   {len(train_dataset)} samples")
print(f"   Validation: {len(val_dataset)} samples")
print(f"   Test:       {len(test_dataset)} samples")
print("="*60)

CREATING DATASETS

Datasets created:
   Training:   108468 samples
   Validation: 19142 samples
   Test:       22520 samples


In [13]:
# Custom Neural Network Architecture - Model 1: LSTM with Attention
class CEFRModel1(nn.Module):
    """
    Architecture 1: Embedding -> LSTM -> Attention -> Fully Connected
    Suitable for sequential text patterns in CEFR classification
    """
    def __init__(self, vocab_size, embedding_dim=128, lstm_hidden=256, num_classes=6, dropout=0.3):
        super(CEFRModel1, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, batch_first=True,
                            bidirectional=True, dropout=dropout if dropout > 0 else 0)

        # Attention layer
        self.attention = nn.Linear(lstm_hidden * 2, 1)
        self.softmax = nn.Softmax(dim=1)

        # Fully connected layers
        self.fc1 = nn.Linear(lstm_hidden * 2, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, input_ids):
        # Embedding
        x = self.embedding(input_ids)  # (batch, seq_len, embedding_dim)
        x = self.dropout(x)

        # LSTM
        lstm_out, _ = self.lstm(x)  # (batch, seq_len, lstm_hidden*2)

        # Attention
        attention_weights = self.attention(lstm_out)  # (batch, seq_len, 1)
        attention_weights = self.softmax(attention_weights)  # Normalize
        context = (lstm_out * attention_weights).sum(dim=1)  # (batch, lstm_hidden*2)

        # Fully connected
        x = self.relu(self.fc1(context))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        logits = self.fc3(x)

        return logits

# Custom Neural Network Architecture - Model 2: CNN + Global Average Pooling
class CEFRModel2(nn.Module):
    """
    Architecture 2: Embedding -> Conv1D -> Global Pooling -> Fully Connected
    Efficient for capturing local patterns in text
    """
    def __init__(self, vocab_size, embedding_dim=128, num_filters=100, num_classes=6, dropout=0.3):
        super(CEFRModel2, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

        # Convolutional layers with different kernel sizes
        self.conv1 = nn.Conv1d(embedding_dim, num_filters, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(embedding_dim, num_filters, kernel_size=4, padding=1)
        self.conv3 = nn.Conv1d(embedding_dim, num_filters, kernel_size=5, padding=2)

        # Fully connected layers
        self.fc1 = nn.Linear(num_filters * 3, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)

        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()

    def forward(self, input_ids):
        # Embedding
        x = self.embedding(input_ids)  # (batch, seq_len, embedding_dim)
        x = self.dropout(x)
        x = x.permute(0, 2, 1)  # (batch, embedding_dim, seq_len)

        # Conv layers
        conv1_out = self.relu(self.conv1(x))  # (batch, num_filters, seq_len)
        conv2_out = self.relu(self.conv2(x))
        conv3_out = self.relu(self.conv3(x))

        # Global average pooling
        pool1 = torch.mean(conv1_out, dim=2)  # (batch, num_filters)
        pool2 = torch.mean(conv2_out, dim=2)
        pool3 = torch.mean(conv3_out, dim=2)

        # Concatenate
        x = torch.cat([pool1, pool2, pool3], dim=1)  # (batch, num_filters*3)

        # Fully connected
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        logits = self.fc3(x)

        return logits

print("Custom neural network architectures defined!")
print("   Model 1: LSTM with Attention")
print("   Model 2: CNN with Global Pooling")

Custom neural network architectures defined!
   Model 1: LSTM with Attention
   Model 2: CNN with Global Pooling


In [14]:
# Training Function
def train_model(model, train_loader, val_loader, device, num_epochs=10, learning_rate=1e-3):
    # Loss function with class weights
    class_weights_tensor = torch.tensor(list(class_weight_dict.values()), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)

    # Optimizer
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                                 patience=2)

    train_losses = []
    val_losses = []
    val_accuracies = []
    best_val_loss = float('inf')
    patience_counter = 0
    max_patience = 3

    for epoch in range(num_epochs):
        # Training
        model.train()
        total_loss = 0
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass
            logits = model(input_ids)
            loss = criterion(logits, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        total_val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids)
                loss = criterion(logits, labels)
                total_val_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        val_acc = accuracy_score(all_labels, all_preds)
        val_accuracies.append(val_acc)

        # Learning rate scheduler
        scheduler.step(avg_val_loss)

        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1

        if (epoch + 1) % 2 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {avg_train_loss:.4f}, " +
                  f"Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

        if patience_counter >= max_patience:
            print(f"Early stopping at epoch {epoch+1}")
            model.load_state_dict(best_model_state)
            break

    return train_losses, val_losses, val_accuracies

# Evaluation Function
def evaluate_model_custom(model, test_loader, device, label_encoder, dataset_name="Test"):
    print(f"\n{'='*60}")
    print(f"EVALUATING ON {dataset_name.upper()} SET")
    print('='*60)

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids)
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    acc = accuracy_score(all_labels, all_preds)
    f1_macro = f1_score(all_labels, all_preds, average='macro')
    f1_weighted = f1_score(all_labels, all_preds, average='weighted')

    print(f"\nOverall Metrics:")
    print(f"   Accuracy:    {acc:.4f} ({acc*100:.2f}%)")
    print(f"   F1 Macro:    {f1_macro:.4f}")
    print(f"   F1 Weighted: {f1_weighted:.4f}")

    # Classification Report
    print(f"\nPer-Class Performance:")
    print("-"*60)
    class_names = label_encoder.classes_
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    # Confusion Matrix
    print(f"\nConfusion Matrix:")
    print("-"*60)
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    print(cm_df)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'predictions': all_preds,
        'true_labels': all_labels
    }

print("Training and evaluation functions ready!")

Training and evaluation functions ready!


---
# 🤖 MODEL 1: LSTM with Attention

Custom architecture optimized for CEFR classification with sequential text patterns.
- **Architecture**: Embedding -> Bidirectional LSTM -> Attention -> Dense Layers
- **Parameters**: ~3-4M
- **Training Time**: ~5-10 minutes on GPU

---

In [15]:
# MODEL 1: Initialize LSTM Model
print("="*60)
print("MODEL 1: LSTM WITH ATTENTION - INITIALIZATION")
print("="*60)

num_classes = len(label_encoder.classes_)
vocab_size = len(tokenizer.word2idx)

model1 = CEFRModel1(
    vocab_size=vocab_size,
    embedding_dim=128,
    lstm_hidden=256,
    num_classes=num_classes,
    dropout=0.3
).to(device)

num_params = sum(p.numel() for p in model1.parameters() if p.requires_grad)

print(f"\nModel 1 created successfully!")
print(f"   Parameters: {num_params:,}")
print(f"   Device: {device}")
print("="*60)

MODEL 1: LSTM WITH ATTENTION - INITIALIZATION

Model 1 created successfully!
   Parameters: 2,236,039
   Device: cuda


In [16]:
# MODEL 1: Create Data Loaders
print("="*60)
print("CREATING DATA LOADERS FOR MODEL 1")
print("="*60)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"\nData loaders created:")
print(f"   Training batches:   {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Test batches:       {len(test_loader)}")
print("="*60)

CREATING DATA LOADERS FOR MODEL 1

Data loaders created:
   Training batches:   3390
   Validation batches: 599
   Test batches:       704


In [17]:
# MODEL 1: Train
print("="*60)
print("TRAINING MODEL 1: LSTM WITH ATTENTION")
print("="*60)

start_time_model1 = time.time()

train_losses1, val_losses1, val_accs1 = train_model(
    model1,
    train_loader,
    val_loader,
    device,
    num_epochs=10,
    learning_rate=0.001
)

training_time_model1 = time.time() - start_time_model1

print("\n" + "="*60)
print("MODEL 1 TRAINING COMPLETE!")
print("="*60)
print(f"Training time: {training_time_model1/60:.2f} minutes")
print("="*60)

TRAINING MODEL 1: LSTM WITH ATTENTION
Epoch [2/10] - Train Loss: 0.3058, Val Loss: 0.2322, Val Acc: 0.9103
Epoch [4/10] - Train Loss: 0.2095, Val Loss: 0.1329, Val Acc: 0.9526
Epoch [6/10] - Train Loss: 0.1482, Val Loss: 0.1333, Val Acc: 0.9553
Epoch [8/10] - Train Loss: 0.1230, Val Loss: 0.1210, Val Acc: 0.9595
Epoch [10/10] - Train Loss: 0.1342, Val Loss: 0.1234, Val Acc: 0.9560
Early stopping at epoch 10

MODEL 1 TRAINING COMPLETE!
Training time: 25.23 minutes


In [18]:
# MODEL 1: Evaluate
model1_results = evaluate_model_custom(
    model1,
    test_loader,
    device,
    label_encoder,
    "Test"
)

model1_results['model_name'] = 'LSTM with Attention'
model1_results['training_time_minutes'] = training_time_model1 / 60


EVALUATING ON TEST SET

Overall Metrics:
   Accuracy:    0.9576 (95.76%)
   F1 Macro:    0.8216
   F1 Weighted: 0.9594

Per-Class Performance:
------------------------------------------------------------
              precision    recall  f1-score   support

          A1     0.0769    0.6667    0.1379        12
          A2     0.9872    0.9673    0.9772      5907
          B1     0.9586    0.9586    0.9586      5914
          B2     0.9654    0.9453    0.9552      5961
          C1     0.9181    0.9540    0.9357      3173
          C2     0.9570    0.9736    0.9652      1553

    accuracy                         0.9576     22520
   macro avg     0.8105    0.9109    0.8216     22520
weighted avg     0.9616    0.9576    0.9594     22520


Confusion Matrix:
------------------------------------------------------------
    A1    A2    B1    B2    C1    C2
A1   8     0     1     2     1     0
A2  51  5714   121    14     7     0
B1  26    66  5669   121    31     1
B2  11     5   110  5635

In [19]:
# MODEL 1: Save
print("\n" + "="*60)
print("SAVING MODEL 1")
print("="*60)

model1_save_path = "./cefr_lstm_attention_model"
import os
os.makedirs(model1_save_path, exist_ok=True)

# Save model
torch.save(model1.state_dict(), f"{model1_save_path}/model_weights.pt")

# Save tokenizer
import pickle
with open(f"{model1_save_path}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open(f"{model1_save_path}/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# Save config
config = {
    'vocab_size': vocab_size,
    'embedding_dim': 128,
    'lstm_hidden': 256,
    'num_classes': num_classes,
    'dropout': 0.3,
    'model_type': 'CEFRModel1'
}

import json
with open(f"{model1_save_path}/config.json", "w") as f:
    json.dump(config, f)

print(f"\nModel 1 saved to: {model1_save_path}")
print(f"   - model_weights.pt")
print(f"   - tokenizer.pkl")
print(f"   - label_encoder.pkl")
print(f"   - config.json")
print("="*60)


SAVING MODEL 1

Model 1 saved to: ./cefr_lstm_attention_model
   - model_weights.pt
   - tokenizer.pkl
   - label_encoder.pkl
   - config.json


---
# ⚡ MODEL 2: CNN with Global Pooling

Efficient custom architecture for local pattern detection in CEFR classification.
- **Architecture**: Embedding -> Multiple Conv1D -> Global Pooling -> Dense Layers
- **Parameters**: ~2-3M
- **Training Time**: ~3-5 minutes on GPU

---

In [20]:
# MODEL 2: Initialize CNN Model
print("="*60)
print("MODEL 2: CNN WITH GLOBAL POOLING - INITIALIZATION")
print("="*60)

model2 = CEFRModel2(
    vocab_size=vocab_size,
    embedding_dim=128,
    num_filters=100,
    num_classes=num_classes,
    dropout=0.3
).to(device)

# Count parameters
num_params2 = sum(p.numel() for p in model2.parameters() if p.requires_grad)

print(f"\nModel 2 created successfully!")
print(f"   Parameters: {num_params2:,}")
print(f"   Device: {device}")
print("="*60)

MODEL 2: CNN WITH GLOBAL POOLING - INITIALIZATION

Model 2 created successfully!
   Parameters: 1,544,626
   Device: cuda


In [21]:
# MODEL 2: Train
print("="*60)
print("TRAINING MODEL 2: CNN WITH GLOBAL POOLING")
print("="*60)

start_time_model2 = time.time()

train_losses2, val_losses2, val_accs2 = train_model(
    model2,
    train_loader,
    val_loader,
    device,
    num_epochs=10,
    learning_rate=0.001
)

training_time_model2 = time.time() - start_time_model2

print("\n" + "="*60)
print("MODEL 2 TRAINING COMPLETE!")
print("="*60)
print(f"Training time: {training_time_model2/60:.2f} minutes")
print("="*60)

TRAINING MODEL 2: CNN WITH GLOBAL POOLING
Epoch [2/10] - Train Loss: 0.3098, Val Loss: 0.5020, Val Acc: 0.8459
Epoch [4/10] - Train Loss: 0.2294, Val Loss: 0.2512, Val Acc: 0.9140
Epoch [6/10] - Train Loss: 0.2013, Val Loss: 0.2909, Val Acc: 0.8995
Epoch [8/10] - Train Loss: 0.1769, Val Loss: 0.2327, Val Acc: 0.9308
Epoch [10/10] - Train Loss: 0.1615, Val Loss: 0.2071, Val Acc: 0.9291

MODEL 2 TRAINING COMPLETE!
Training time: 15.63 minutes


In [22]:
# MODEL 2: Evaluate
model2_results = evaluate_model_custom(
    model2,
    test_loader,
    device,
    label_encoder,
    "Test"
)

model2_results['model_name'] = 'CNN with Global Pooling'
model2_results['training_time_minutes'] = training_time_model2 / 60


EVALUATING ON TEST SET

Overall Metrics:
   Accuracy:    0.9251 (92.51%)
   F1 Macro:    0.8096
   F1 Weighted: 0.9255

Per-Class Performance:
------------------------------------------------------------
              precision    recall  f1-score   support

          A1     0.1270    0.6667    0.2133        12
          A2     0.9160    0.9873    0.9503      5907
          B1     0.9493    0.8793    0.9129      5914
          B2     0.9268    0.9195    0.9231      5961
          C1     0.8908    0.8869    0.8888      3173
          C2     0.9734    0.9646    0.9690      1553

    accuracy                         0.9251     22520
   macro avg     0.7972    0.8840    0.8096     22520
weighted avg     0.9276    0.9251    0.9255     22520


Confusion Matrix:
------------------------------------------------------------
    A1    A2    B1    B2    C1    C2
A1   8     0     0     3     1     0
A2  26  5832    20    18    11     0
B1  10   525  5200   100    76     3
B2  14     5   243  5481

In [23]:
# MODEL 2: Save
print("\n" + "="*60)
print("SAVING MODEL 2")
print("="*60)

model2_save_path = "./cefr_cnn_model"
os.makedirs(model2_save_path, exist_ok=True)

# Save model
torch.save(model2.state_dict(), f"{model2_save_path}/model_weights.pt")

# Save tokenizer
with open(f"{model2_save_path}/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# Save label encoder
with open(f"{model2_save_path}/label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

# Save config
config2 = {
    'vocab_size': vocab_size,
    'embedding_dim': 128,
    'num_filters': 100,
    'num_classes': num_classes,
    'dropout': 0.3,
    'model_type': 'CEFRModel2'
}

with open(f"{model2_save_path}/config.json", "w") as f:
    json.dump(config2, f)

print(f"\nModel 2 saved to: {model2_save_path}")
print(f"   - model_weights.pt")
print(f"   - tokenizer.pkl")
print(f"   - label_encoder.pkl")
print(f"   - config.json")
print("="*60)


SAVING MODEL 2

Model 2 saved to: ./cefr_cnn_model
   - model_weights.pt
   - tokenizer.pkl
   - label_encoder.pkl
   - config.json


---
# 📊 MODEL COMPARISON

Comparing custom architectures for CEFR classification

---

In [24]:
# Compare Models
print("="*80)
print(" " * 20 + "CUSTOM MODELS COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Model': ['LSTM with Attention', 'CNN with Global Pooling'],
    'Accuracy (%)': [
        model1_results['accuracy'] * 100,
        model2_results['accuracy'] * 100
    ],
    'F1 Macro': [
        model1_results['f1_macro'],
        model2_results['f1_macro']
    ],
    'F1 Weighted': [
        model1_results['f1_weighted'],
        model2_results['f1_weighted']
    ],
    'Training Time (min)': [
        model1_results['training_time_minutes'],
        model2_results['training_time_minutes']
    ],
    'Parameters': [
        num_params,
        num_params2
    ]
})

print("\n" + comparison_df.to_string(index=False))

print("\n" + "="*80)
if model1_results['accuracy'] > model2_results['accuracy']:
    winner = 'LSTM with Attention'
    diff = (model1_results['accuracy'] - model2_results['accuracy']) * 100
else:
    winner = 'CNN with Global Pooling'
    diff = (model2_results['accuracy'] - model1_results['accuracy']) * 100

print(f"🏆 BEST MODEL: {winner}")
print(f"   Accuracy advantage: {diff:.2f}%")

if model1_results['training_time_minutes'] < model2_results['training_time_minutes']:
    faster = 'LSTM with Attention'
    time_diff = model2_results['training_time_minutes'] - model1_results['training_time_minutes']
else:
    faster = 'CNN with Global Pooling'
    time_diff = model1_results['training_time_minutes'] - model2_results['training_time_minutes']

print(f"\nFASTER MODEL: {faster}")
print(f"   Time saved: {time_diff:.2f} minutes")

if num_params < num_params2:
    smaller = 'LSTM with Attention'
    param_diff = num_params2 - num_params
else:
    smaller = 'CNN with Global Pooling'
    param_diff = num_params - num_params2

print(f"\nSMALLER MODEL: {smaller}")
print(f"   Parameters saved: {param_diff:,}")

print("="*80)

                    CUSTOM MODELS COMPARISON

                  Model  Accuracy (%)  F1 Macro  F1 Weighted  Training Time (min)  Parameters
    LSTM with Attention     95.759325  0.821639     0.959368            25.233342     2236039
CNN with Global Pooling     92.508881  0.809574     0.925520            15.631804     1544626

🏆 BEST MODEL: LSTM with Attention
   Accuracy advantage: 3.25%

FASTER MODEL: CNN with Global Pooling
   Time saved: 9.60 minutes

SMALLER MODEL: CNN with Global Pooling
   Parameters saved: 691,413


In [25]:
# Load Models for Inference
def load_model(model_path, model_type):
    """Load saved model"""
    import json

    # Load config
    with open(f"{model_path}/config.json", "r") as f:
        config = json.load(f)

    # Load tokenizer and label encoder
    with open(f"{model_path}/tokenizer.pkl", "rb") as f:
        tok = pickle.load(f)

    with open(f"{model_path}/label_encoder.pkl", "rb") as f:
        encoder = pickle.load(f)

    # Create model
    if config['model_type'] == 'CEFRModel1':
        model = CEFRModel1(
            vocab_size=config['vocab_size'],
            embedding_dim=config['embedding_dim'],
            lstm_hidden=config['lstm_hidden'],
            num_classes=config['num_classes'],
            dropout=config['dropout']
        )
    else:
        model = CEFRModel2(
            vocab_size=config['vocab_size'],
            embedding_dim=config['embedding_dim'],
            num_filters=config['num_filters'],
            num_classes=config['num_classes'],
            dropout=config['dropout']
        )

    # Load weights
    model.load_state_dict(torch.load(f"{model_path}/model_weights.pt", map_location=device))
    model = model.to(device)
    model.eval()

    return model, tok, encoder

# Inference Function
def predict_cefr_level(text, model_path, model_choice='model1'):
    """
    Predict CEFR level for a new paragraph

    Args:
        text (str): The input paragraph
        model_path (str): Path to the saved model
        model_choice (str): 'model1' or 'model2'

    Returns:
        dict: Prediction results with label and confidence
    """
    # Load model and components
    model, tok, encoder = load_model(model_path, model_choice)

    # Encode text
    input_ids = tok.encode(text)
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Predict
    with torch.no_grad():
        logits = model(input_tensor)
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0][predicted_class].item()

    # Get label name
    predicted_label = encoder.classes_[predicted_class]

    # Get all probabilities
    all_probs = {
        encoder.classes_[i]: probabilities[0][i].item()
        for i in range(len(encoder.classes_))
    }

    return {
        'predicted_level': predicted_label,
        'confidence': confidence,
        'all_probabilities': all_probs
    }

print("Inference functions ready!")

Inference functions ready!


In [26]:
# Test Inference with Examples
print("="*80)
print("TESTING INFERENCE FUNCTIONS")
print("="*80)

# Test examples from different CEFR levels
test_examples = [
    "I like cats. They are nice. I have one cat.",  # Simple sentence (A1-A2)
    "Yesterday I went to the store and bought some groceries. It was a pleasant experience.",  # Intermediate (B1)
    "The implementation of sustainable practices in modern corporations requires a comprehensive understanding of both environmental and economic factors.",  # Advanced (C1-C2)
]

print("\n" + "-"*80)
for i, example in enumerate(test_examples, 1):
    print(f"\nExample {i}: {example[:80]}...")

    # Predict with both models
    try:
        model1_pred = predict_cefr_level(example, model1_save_path, 'model1')
        model2_pred = predict_cefr_level(example, model2_save_path, 'model2')

        print(f"\n  LSTM Model:   {model1_pred['predicted_level']} (confidence: {model1_pred['confidence']:.3f})")
        print(f"  CNN Model:    {model2_pred['predicted_level']} (confidence: {model2_pred['confidence']:.3f})")
    except Exception as e:
        print(f"  Error during inference: {e}")

    print("-"*80)

print("\nCustom models are ready for inference!")
print("="*80)

TESTING INFERENCE FUNCTIONS

--------------------------------------------------------------------------------

Example 1: I like cats. They are nice. I have one cat....

  LSTM Model:   A1 (confidence: 0.999)
  CNN Model:    A2 (confidence: 1.000)
--------------------------------------------------------------------------------

Example 2: Yesterday I went to the store and bought some groceries. It was a pleasant exper...

  LSTM Model:   A2 (confidence: 1.000)
  CNN Model:    A2 (confidence: 1.000)
--------------------------------------------------------------------------------

Example 3: The implementation of sustainable practices in modern corporations requires a co...

  LSTM Model:   A2 (confidence: 1.000)
  CNN Model:    A2 (confidence: 1.000)
--------------------------------------------------------------------------------

Custom models are ready for inference!


---
# ✅ TRAINING COMPLETE!

Both custom models have been trained and evaluated:
- **LSTM with Attention**: Captures sequential patterns with attention mechanism
- **CNN with Global Pooling**: Detects local text patterns efficiently

## 📁 Saved Files:
- `./cefr_lstm_attention_model/` - LSTM model with attention
  - model_weights.pt
  - tokenizer.pkl
  - label_encoder.pkl
  - config.json

- `./cefr_cnn_model/` - CNN model with global pooling
  - model_weights.pt
  - tokenizer.pkl
  - label_encoder.pkl
  - config.json

## 🚀 Next Steps:
1. Review the comparison results above
2. Choose the best performing custom model
3. Use `predict_cefr_level()` function for new paragraphs
4. Deploy the custom model for production use

## 📊 Key Features:
- ✅ Custom architecture designed for CEFR classification
- ✅ Lightweight models with 2-4M parameters
- ✅ Fast training on GPU (3-10 minutes)
- ✅ Handles class imbalance with weighted loss
- ✅ Vocabulary-based tokenization
- ✅ Inference function for predictions

---